This notebook contains medical note generation from approach 1: Metadata --> Note --> Dialogue

## Imports

In [1]:
import pandas as pd
import datetime
import openai
from datetime import datetime
import os
import csv
from openai import OpenAI
import random

## Classes and functions

In [2]:
# provides samples from Aci Bench dataset

class AciExampleProvider:
    def __init__(self, data_file, sample_size=15):
        self.data_file = data_file
        self.sample_size = sample_size
        self.meta_and_note = ""

    def sample_meta_and_note(self):
        """
        Samples metadata and note content from Aci.
        """
        try:
            aci_df = pd.read_csv(self.data_file)
            aci_df.rename(columns={'cc': 'chief_complaint'}, inplace=True)
            aci_data = aci_df[aci_df['dataset'] == 'aci']
            aci_data = aci_data.sample(n=self.sample_size)

            example_number = 1
            for index, row in aci_data.iterrows():
                metadata = row[-7:]
                note = row['note']

                self.meta_and_note += f"Example {example_number}:\n\nMetadata:\n"
                for col, value in metadata.items():
                    self.meta_and_note += f" {col}: {value}, "
                self.meta_and_note = self.meta_and_note.rstrip(', ')
                self.meta_and_note += f"\n\n\n\nNote:\n{note}\n\n{'*'*75}\n\n"
                example_number += 1

        except Exception as e:
            return f"An error occurred while sampling meta and note: {e}"
        
        return self.meta_and_note

    def write_to_file(self, output_file_base):
        """
        Just in case we wanna save the sampled data. 
        Writes the sampled meta and note to a file with a date-stamped filename.
        """
        current_date = datetime.datetime.now().strftime("%d%m")
        output_file = f"{output_file_base}_{current_date}.txt"
        try:
            with open(output_file, 'w') as file:
                file.write(self.meta_and_note)
            return f"Data has been written to {output_file}"
        except Exception as e:
            return f"An error occurred: {e}"

In [3]:

# to open a file and return its content as a string:
def open_file(file_path):
  with open(file_path, "r", encoding = "utf-8") as infile:
    return infile.read()
  

def save_to_csv2(data, base_filename, directory):
    # Ensure the directory ends with a slash
    if not directory.endswith('/'):
        directory += '/'

    # Get the current date in YYYYMMDD format
    current_date = datetime.now().strftime("%Y%m%d")

    # Create the filename with the current date and directory
    filename = f"{directory}{base_filename.split('.')[0]}_{current_date}.csv"

    # Save the data to a CSV file
    with open(filename, mode='w', newline='', encoding='utf-8') as file:
        writer = csv.writer(file)
        writer.writerow(['Diagnosis_Desc', 'Synthetic_Note'])
        for row in data:
            writer.writerow(row)

    print(f"Data has been saved to {filename}")

  
def initialize_openai_client(api_key_file):
    """
    Initialize the OpenAI client with an API key read from a file.

    Args:
    api_key_file (str): The path to the file containing the OpenAI API key.

    Returns:
    openai.Client: An instance of the OpenAI client.
    """
    try:
        with open(api_key_file, 'r') as file:
            api_key = file.read().strip()
        client = openai.Client(api_key=api_key)
        return client
    except Exception as e:
        print(f"An error occurred: {e}")
        return None
    




## Note Generation:

### Preparations

In [4]:
def generate_text_no_system(PROMPT):
    response = client.chat.completions.create(
        model="gpt-4-1106-preview",
        messages=[
            {"role": "user", "content": PROMPT}
        ],
        temperature=0.7
    )
    return response.choices[0].message.content.strip()

In [5]:
def generate_text_with_system(PROMPT):
    response = client.chat.completions.create(
        model="gpt-4-1106-preview",
        messages=[
            {
            "role": "system",
            "content": ""
            },
            {"role": "user", "content": PROMPT}
        ],
        temperature=0.7
    )
    return response.choices[0].message.content.strip()

In [6]:
# In case we saved the samples from Aci into a csv and want to read it, this is useful
# Read the contents of the example file into a variable
''' 
with open('/Users/ahmadrezaie/DalPhD/Research/note_taking/Data/sample_note_and_dialog.txt', 'r') as file:
    examples = file.read()

''' 

" \nwith open('/Users/ahmadrezaie/DalPhD/Research/note_taking/Data/sample_note_and_dialog.txt', 'r') as file:\n    examples = file.read()\n\n"

In [7]:
# icd distibution data:

icd_df = pd.read_csv("/Users/ahmadrezaie/DalPhD/Research/note_taking/Data/IQVIA/ICD10count_diag_admit.csv")


In [8]:
# List of inclusive names from different races
names = ["Aisha", "Carlos", "Wei", "Fatima", "John", "Olga", "Dev", "Yuki", "Nia", "Samuel", "Chen", "Irina", "Mohammed", "Leila", "Raj", "Lena", "Hiroshi", "Tanya", "Luis", "Zara"]


# initializing openAI:
client = initialize_openai_client("/Users/ahmadrezaie/DalPhD/Research/note_taking/Code/OpenAIkey.txt")


### Note Generation Loop

In [ ]:
# propmpt:

PROMPT= """ I am giving you some examples of doctor's notes from visiting a patient. The notes re generated from some metadata.
            The examples contain the metadata and the correscponding note.
            Your task is to generate a similar structure note but for a differrent medical condition. Please try your
            best. It is very important for the patients and their families.
            Please have in mind that you are given the chief_complaint in the examples, but I will give you the 
            "ICD-10_code_description" of a disease and you should infer the "chief_complaint" yourself to generate the note. 
            Also, please consider a secondary complaint yourself based on what you think is relevant and approprite. 
            Here is the new metadata:
            "ICD-10_code_description: "{icd_description}", doctor_name: {doctor_name}, patient_gender: {gender}, patient_age: {age}, patient_firstname: {first_name}, patient_familyname: {family_name}
            Here are the example: {example}
            """

In [ ]:
all_generated_notes = []

try:
        for i, row in icd_df.head(10).iterrows():
            # Randomly select gender, age, and names
            gender = random.choice(["male", "female"])
            age = str(random.randint(18, 75))
            first_name = random.choice(names)
            family_name = random.choice(names)
            doctor_name = random.choice(names)  

            modified_prompt = PROMPT.format(
                icd_description=row['diagnosis_desc'],
                doctor_name=doctor_name,
                gender=gender,
                age=age,
                first_name=first_name,
                family_name=family_name,
                example=examples
                )

            synthetic_note = generate_text_no_system(modified_prompt)
            if i <= 2:
                print(row['diagnosis_desc'], synthetic_note)

            all_generated_notes.append([row['diagnosis_desc'], synthetic_note])
            print(f"Completed API call {i + 1}")
            
except KeyboardInterrupt:
        print("\nKeyboard interrupt detected. Saving partial data to the CSV file...")

finally:
        save_to_csv2(all_generated_notes, "meta_to_note_random_age_gender_sample_examples.csv", "/Users/ahmadrezaie/DalPhD/Research/note_taking/Data/")

## Check the rersults

print()

# Further tests:

## With system prompt, no Aci data used:

In [6]:
# initializing openAI:
client = initialize_openai_client("/Users/ahmadrezaie/DalPhD/Research/note_taking_v2/data/input/OpenAIkey.txt")


An error occurred: [Errno 2] No such file or directory: '/Users/ahmadrezaie/DalPhD/Research/note_taking_v2/data/input/OpenAIkey.txt'


### Single LLM doctor

In [7]:
system_prompt =  "Assume you are a very experienced family physician doctor and you are conducting a research. The research project is to generate synthetic medical notes from doctor-patient conversations. \nThe notes must in this format:\n\n1. Subjective: This section includes the patient's own description of their symptoms and complaints.\n\n2. Objective: This section includes observations and data gathered by the family physician, such as vital signs, physical examination findings, and test results.\n\n3. Assessment: This section includes the family physician's evaluation of the patient's condition, including a diagnosis or differential diagnosis.\n\n4. Plan: This section includes the family physician's recommendations for treatment, management, and follow-up. \n\n\nThe notes must contain these variables:\n\n1) Medical Outcome: Like diagnosis, prescribed treatment, follow-up recommendations, referral to specialists, referral to further tests or imaging, medication adjustment, lifestyle change (sleep, diet, exercise, tobacco use, alcohol use)\n\n\n2) Medical History: Like previous diagnoses, family medical history, medication history, allergies, and chronic conditions.\n\n3) Symptom Description: Like severity, duration, associated symptoms, frequency, and impact on daily activities.\n\n4) Patient’s self-reported habits and lifestyle: Sleep, diet, exercise, tobacco use, alcohol consumption, drug use, recreational activities.\n\n5) Demographic Information: Like age, gender, ethnicity, socio-economic status, education level, health literacy, and job status.\n\n6) Patient's Behavior: Like the patient's cooperation with medical advice\n\n7) Geographical Location: Like Big city vs small city, rural vs urban, pollution and environmental health risks, neighborhood type- eg if impoverished or affluent, well-served by transit, food desert, etc.\n\n8) Clinical Setting: Like hospitals, clinics, telemedicine, community health services, urgent care centers, research facilities, school health services, private practice, and specialty clinics.\n\n9) Type of Encounter: Like initial consultation, follow-up, emergency visit, routine check-up, chronic disease management (regular appointments to manage long-term health conditions like diabetes, heart disease, or chronic pain), preventive health screening.\n\n10) Treatment Disparities: There may be a tendency to offer less aggressive treatment or fewer options due to assumptions about compliance or ability to pay.\n\n11) Native or Non-Native English Speaking Patient.\n\nThe user will give you the ICD-10 description of the disease and you should generate a note in the format mentioned above. "

def generate_note(condition):
        response = client.chat.completions.create(
        model="gpt-4-1106-preview",
        messages=[
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": condition
            }
        ],
        temperature=0.68,
        max_tokens=4052,
        top_p=1,
        frequency_penalty=0,
        presence_penalty=0
        )
        return response.choices[0].message.content

In [13]:
note = generate_note("Hypertention")
print(note)

# ask the  model to write down the gist of the scenario, and the values of variables its going to use. Also, write down
# every medical fact its going to use. It will make it easier to evaluate in the future and make it easier to ensure the variety of notes for 
# a medical condition. The scenarios are shorter to feed to the model iteratively.

Subjective:
Mr. Johnson, a 52-year-old African American male, presents with complaints of occasional headaches and dizziness over the past month. He describes the headaches as moderate in intensity, mostly occurring in the late afternoon, and sometimes accompanied by a sensation of pulsating in his temples. He denies any associated nausea or visual disturbances. Mr. Johnson reports the symptoms have been impacting his concentration at work as an accountant. He has a history of hypertension diagnosed two years ago. His father also had hypertension. He's currently on hydrochlorothiazide but admits to missing doses occasionally.

Objective:
Vital Signs: BP 162/98 mmHg, HR 78 bpm, RR 16 breaths/min, Temp 98.6°F (37°C).
Physical Examination: Auscultation reveals no abnormal heart or lung sounds. BMI is 29, indicating overweight status. No edema is noted in extremities. Fundoscopic exam shows no signs of papilledema or vascular changes.
Tests: Previous lab results from 3 months ago showed no

In [8]:
note = generate_note("Hypertention")
print(note)

Subjective:
A 53-year-old male patient presents for a routine follow-up appointment complaining of occasional headaches and dizziness over the past two months. He reports that these symptoms often occur in the morning and sometimes after physical exertion. The patient notes that his home blood pressure readings have been higher than usual, averaging around 145/95 mmHg. He denies chest pain, palpitations, or dyspnea. He mentions his father had hypertension and his mother has type 2 diabetes. The patient is a long-term smoker and admits to a sedentary lifestyle with a high-sodium diet.

Objective:
The patient is alert and oriented. Blood pressure is measured at 148/92 mmHg in the office. Heart rate is regular at 78 bpm, and respiratory rate is 16 breaths per minute. BMI is calculated at 30, indicating obesity. Physical examination reveals no additional abnormalities. No edema is noted. Fundoscopic exam is unremarkable with no signs of hypertensive retinopathy. Laboratory tests including 

In [10]:
# main loop to ask for medical condition, then generate notes:
import datetime

# Get the current date and format it as a string
current_date = datetime.datetime.now().strftime("%Y-%m-%d")

# Create the filename with the current date
path = "/Users/ahmadrezaie/DalPhD/Research/note_taking_v2/data/output"
filename = f"{path}/medical_notes_{current_date}.csv"

# Main loop
with open(filename, 'w', newline='', encoding='utf-8') as file:
    writer = csv.writer(file)
    
    # Write the prompt at the top of the file
    writer.writerow(["Prompt: " + system_prompt])
    
    # Write column headers
    writer.writerow(["Medical_Condition", "Note"])

    while True:
        # Ask the user for a medical condition
        condition = input("Enter a medical condition (or type 'exit' to stop): ").strip()
        
        # Check if the user wants to exit
        if condition.lower() == 'exit':
            break
        
        # Generate medical notes for the given condition
        medical_notes = generate_note(condition)

        # Write the condition and notes to the CSV file
        writer.writerow([condition, medical_notes])
        print("Note saved to CSV.")

print("Exiting the program.")

Note saved to CSV.
Note saved to CSV.
Note saved to CSV.
Note saved to CSV.
Note saved to CSV.
Note saved to CSV.
Note saved to CSV.
Note saved to CSV.
Note saved to CSV.
Note saved to CSV.
Note saved to CSV.
Note saved to CSV.
Note saved to CSV.
Note saved to CSV.
Note saved to CSV.
Note saved to CSV.
Note saved to CSV.
Note saved to CSV.
Exiting the program.


The model did not get it when I asked to generate 5 different notes for a single condition. 

How to make sure to make the notes as different as possible:

1) Use Prompt Modifiers: Integrate modifiers in your prompts to guide the generation of diverse notes. For example:

"Write a medical note focusing on the treatment plan for [condition]."
"Create a brief note emphasizing the diagnostic process for a patient with [condition]."

We can use "Medical Outcome" control variable for this. Like:
Write a medical note focusing on "[medicaloutcome_1]" for [condition].
Write a medical note focusing on "[medicaloutcome_2]" for [condition].
Write a medical note focusing on "[medicaloutcome_1]" and "[medicaloutcome_2]" for [condition].


2) Controlled Randomness: Introduce controlled randomness. You could randomize certain elements of your prompt (like patient age, symptoms, or background) using your own code before sending it to the API.

3) Sequential Requests with Memory: Make sequential requests where each new request builds upon or differs from the previous ones. You can maintain a 'memory' or log of past requests and guide the API to generate notes that are different. 


4) Feedback Loop: Use a feedback loop where the output of the previous generation influences the next prompt. This could be automated or manual, where you assess the diversity and tweak the prompts accordingly.
I am using the cobmination of 3 and 4:

It means we can iteratively add the model's previous generated notes to the input prompt and ask to make it different thn this one. 
Look at this link also: https://community.openai.com/t/function-calling-with-memory/307051 


5) Having another LLM as "Judge" to check if the generated notes are different.

Test this approach:

Have 2 LLMs; The doctor and the Judge. The doctor first generates the scenario (the values pf each variable that it is going to use to generate the notes) and pass it to the Judge. Judge then compares it with the previous scenarios (combinations of variables) for the same ICD-10. If at least 4 out of 11 variables are differernt, then the Judge outputs "Go", and the doctor proceeds to generate the note. If the condition is not met, the Judge outputs "No Go", and then the doctor should revise the scenario to pass the condition. 
We can test this approach for 15 ICD-10 codes, generating 15 notes for each. Then, check the similarity score (with some embeding approach) between the notes from the same ICD-10 codes to test if the approach works. 




### Three LLM setting: Scenario provider, Judge, and note generator:

Test this approach:

Have 2 LLMs; The doctor and the Judge. The doctor first generates the scenario (the values pf each variable that it is going to use to generate the notes) and pass it to the Judge. Judge then compares it with the previous scenarios (combinations of variables) for the same ICD-10. If at least 4 out of 11 variables are differernt, then the Judge outputs "Go", and the doctor proceeds to generate the note. If the condition is not met, the Judge outputs "No Go", and then the doctor should revise the scenario to pass the condition. 
We can test this approach for 15 ICD-10 codes, generating 15 notes for each. Then, check the similarity score (with some embeding approach) between the notes from the same ICD-10 codes to test if the approach works. 

##### implementation

In [4]:
model = "gpt-4-1106-preview"
temperature=1
max_tokens=4052
top_p=1
frequency_penalty=0
presence_penalty=0


doctor_scenario_generator_system_prompt ="Assume you are a very experienced physician and you are conducting research. The research project is to generate synthetic medical notes from doctor-patient conversations. \n\nThe notes must contain these variables:\n\n1) Medical Outcome: Like diagnosis, prescribed treatment, follow-up recommendations, referral to specialists, referral to further tests or imaging, medication adjustment, lifestyle change (sleep, diet, exercise, tobacco use, alcohol use). If it is prescribing medication, details should be included like dose, units, frequency, duration, quantity, quantity type (like tablets, etc.), and route (like oral or injected). If it is a referral, it should include details like the reason for the referral, the specialty, and the doctor's name. If it is an order for blood work, it should include details like if it is for biochemistry, hematology, immunology, microbiology, viral hepatitis, vitamin D, prostate-specific antigen, or anything else that suits the scenario. You need to be very specific. If it's an order for imaging, it should include details like the modality of the imaging and the area of the body. For example, if it is ultrasound, it can be an order for Abdominal, Thyroid, Musculoskeletal, Sonohysterogram, Sonohysterogram, Biophysical Profile(BPP), Scrotal, G.U. Tract - Kidneys-Bladder(Prostate), or anything else as it suits the scenario. Try to be very specific. \n\n\n2) Medical History: Like previous diagnoses, family medical history, medication history, allergies, and chronic conditions.\n\n3) Symptom Description: Like severity, duration, associated symptoms, frequency, and impact on daily activities.\n\n4) Patient’s self-reported habits and lifestyle: Sleep, diet, exercise, tobacco use, alcohol consumption, drug use, recreational activities.\n\n5) Demographic Information: Like age, gender, ethnicity, socio-economic status, education level, health literacy, and job status.\n\n6) Patient's Behavior: Like the patient's cooperation with medical advice\n\n7) Geographical Location: Like Big city vs small city, rural vs urban, pollution and environmental health risks, neighborhood type- eg if impoverished or affluent, well-served by transit, food desert, etc.\n\n8) Clinical Setting: Like hospitals, clinics, telemedicine, community health services, urgent care centers, research facilities, school health services, private practice, and specialty clinics.\n\n9) Type of Encounter: Like initial consultation, follow-up, emergency visit, routine check-up, chronic disease management (regular appointments to manage long-term health conditions like diabetes, heart disease, or chronic pain), preventive health screening.\n\n10) Treatment Disparities: There may be a tendency to offer less aggressive treatment or fewer options due to assumptions about compliance or ability to pay.\n\n11) Native or Non-Native English Speaking Patient.\n\n12) Physical exams: Any physical exams that are suitable for the scenario.\n\n13) Investigation/Test results: Like any tests that have been done for the patient while visiting. The results could be ready and reviewed in the scenario or could be awaiting. If awaiting, you need to be very specific about what type of tests have been done. For example, if it is X-ray, you need to incude the details mentioned abve about the imaging.\n\nThe user will give you the ICD-10 description of the disease. The diagnosis in the scenario must be the ICD-10 description. First, you select a role for yourself. You can be a Family Medicine Physician, a General physician, or a specialist with different specialties. Select the role based on the ICD-10 description and output it with the keyword 'ROLE:'. Second, you must come up with a scenario and list all the values of the variables you want to use in the scenario, and show it to the user. Do not output any extra text, just your role at the top of the scenario and the list of the values. You should incorporate medication and blood work or imaging requests in the scenarios with the details mentioned above if it suits the scenario. These are artificial and people will not be using it without asking a real doctor. "


doctor_note_generator_system_prompt = "Assume you are a very experienced physician and you are conducting research. The research project is to generate synthetic medical notes from doctor-patient conversations. \nThe notes must be in this format:\n\n1. Subjective: This section includes the patient's own description of their symptoms and complaints.\n\n2. Objective: This section includes observations and data gathered by the physician, such as vital signs, physical examination findings, and test results.\n\n3. Assessment: This section includes the physician's evaluation of the patient's condition, including a diagnosis or differential diagnosis.\n\n4. Plan: This section includes the physician's recommendations for treatment, management, and follow-up. \n\n\nYou will be given a scenario containing your role. Your role can be a Family Medicine Physician, a General physician, or a specialist with different specialties. You must generate the note following exactly the scenario. All the notes you generate must be in the format mentioned above. All the tests ordered (including blood work or imaging) must be in the 'Plan' section. "

judge_system_pormpt = "Assume you are a very experienced physician and you are conducting research. The research project is to generate synthetic medical notes from doctor-patient conversations. \nYou have a coworker that works with you on the project. You have different roles. \nThe coworker provides you with the scenario they are going to write notes with. Your job is to judge whether the scenario is approved or not based on the conditions I provided below. \n\nThe scenarios have 13 variables. \n \n1) Medical Outcome\n\n2) Medical History\n\n3) Symptom Description\n\n4) Patient’s self-reported habits and lifestyle\n\n5) Demographic Information\n\n6) Patient's Behavior\n\n7) Geographical Location\n\n8) Clinical Setting\n\n9) Type of Encounter\n\n10) Treatment Disparities\n\n11) Native or Non-Native English Speaking Patient.\n\n12) Physical exams: Any physical exams that are suitable for the scenario.\n\n13) Investigation/Test results: Like any tests that have been done for the patient while visiting. The results could be ready and reviewed in the scenario or could be awaiting.\n\n\nYou have to check three conditions and then decide to approve or deny the scenario:\n\na) A pair-wise comparison of the values of the variables. You need to check if at least 5 out of 13 of the values of the variables in the scenario are different from the previously approved scenarios.\n\nb) You need to check if the scenario is medically correct in terms of symptoms, tests, diagnosis, and treatment. \n\nc) You need to check if the scenario is plausible or not.\n\nFirst, check condition \"a\". If it is not met, the scenario is rejected and you do not need to check conditions \"b\" and \"c\". \n\nIf the scenario passes these three conditions, then say \"Go\". If not, you say \"NoGo\".  In the case that there is no scenario previously approved, you should only check conditions \"b\" and \"c\".\n\n\"Go\" or \"No Go\" must be your only words. "

note_polisher_system_prompt =  "Assume you are a very experienced physician and you are conducting research. The research project is to generate synthetic medical notes from doctor-patient conversations. \nThe notes must be in this format:\n\n1. Subjective: This section includes the patient's own description of their symptoms and complaints.\n\n2. Objective: This section includes observations and data gathered by the physician, such as vital signs, physical examination findings, and test results.\n\n3. Assessment: This section includes the physician's evaluation of the patient's condition, including a diagnosis or differential diagnosis.\n\n4. Plan: This section includes the physician's recommendations for treatment, management, and follow-up. \n\n\nYou will be given a note. Your task is to polish the note and make sure the information is placed correctly in the relevant section.  You cannot add or remove any information, except where you have been given permission.\n\nMake sure of these:\n\na) If the doctor is ordering an imaging or bloodwork to be done, it must come under the Plan\" section. But if it is already done, it can come in other sections. \n\nb) If the doctor is prescribing a medication or renewing a medication, changing doses, etc., it must be under the \"Plan\" section. \n\nc) Referrals must come under the \"Plan\" section. The referral must contain the reason for referral, the specialty, and the doctor's name. If any part is missing, add it. You can choose any appropriate name, do not stick with one name.\n\nd) Patients' must have names. If there is no name, add it. You can choose any appropriate name, do not stick with one name.\n\n\nJust output the revised note, not anything else. \n"


In [7]:
# initializing openAI:
client = initialize_openai_client("/h/ahmad/SynthDataGen/Synthetic_Data_Gen/data/input/OpenAIkey.txt")


In [7]:

def doctor_generate_scenario(condition):
        scenario_response = client.chat.completions.create(
        model=model,
        temperature = temperature,
        max_tokens = max_tokens,
        top_p = top_p,
        frequency_penalty = frequency_penalty,
        presence_penalty = presence_penalty,
        messages=[
            {
                "role": "system",
                "content": doctor_scenario_generator_system_prompt
            },
            {
                "role": "user",
                "content": condition
            }
        ]
        )
        return scenario_response.choices[0].message.content



def doctor_generate_note(scenario):
        note_response = client.chat.completions.create(
        model=model,
        temperature = temperature,
        max_tokens = max_tokens,
        top_p = top_p,
        frequency_penalty = frequency_penalty,
        presence_penalty = presence_penalty,
        messages=[
            {
                "role": "system",
                "content": doctor_note_generator_system_prompt
            },
            {
                "role": "user",
                "content": scenario
            }
        ]
        )
        return note_response.choices[0].message.content


def judge_evaluate_scenario(scenario, conversations_memory):
    # Add system prompt and user scenario to memory
        conversations_memory += [
        {"role": "user", "content": scenario}
        ]
        evaluation_response = client.chat.completions.create(
        model=model,
        temperature = temperature,
        max_tokens = max_tokens,
        top_p = top_p,
        frequency_penalty = frequency_penalty,
        presence_penalty = presence_penalty,
        messages=conversations_memory
        )
        # Accessing the last message's content correctly
        latest_message = evaluation_response.choices[0].message.content
        print("Latest message is: ", latest_message)
        decision = latest_message.split()[-1]  # Extract the last word

        # Update memory with the model's latest response
        conversations_memory.append({"role": "assistant", "content": latest_message})
        print(decision)
        print(conversations_memory)
        print("len of conversation_momory is: ",len(conversations_memory))
        return decision

In [8]:
def polish(note):
        note_response = client.chat.completions.create(
        model=model,
        temperature = temperature,
        max_tokens = max_tokens,
        top_p = top_p,
        frequency_penalty = frequency_penalty,
        presence_penalty = presence_penalty,
        messages=[
            {
                "role": "system",
                "content": note_polisher_system_prompt
            },
            {
                "role": "user",
                "content": note
            }
        ]
        )
        return note_response.choices[0].message.content


In [9]:
# a function to extract the "ROLE" from the scenario:

import re

def extract_role(text):
    # take the first 5 lines
    first_5_lines = '\n'.join(text.splitlines()[:5])
    # Define the regular expression pattern to find "ROLE:" (in any capitalization) followed by any characters until a line break
    pattern = r"ROLE: (.+?)\n"

    match = re.search(pattern, first_5_lines, re.IGNORECASE)
    # If a match is found, return the group which matches the role description
    if match:
        return match.group(1)
    else:
        # Return None or an appropriate message if "ROLE:" is not found within the first 5 lines
        return "Role not found in the first 5 lines."


In [10]:
def generate_medical_notes(disease_description, notes_count):
    approved_notes = []
    rejected_scenarios = []
    judge_conversations_memory = [{"role": "system", "content": judge_system_pormpt}]

    try:
        while True:
            scenario = doctor_generate_scenario(disease_description)
            decision = judge_evaluate_scenario(scenario, judge_conversations_memory)
            if decision == "Go" or decision == "Go.":
                role = extract_role(scenario)
                note = doctor_generate_note(scenario)
                polished_note = polish(note)
                approved_notes.append({"Disease Description": disease_description, "Scenario": scenario, "Note": note, "Polished Note": polished_note , "Role": role })
            else:
                rejected_scenarios.append({"Disease Description": disease_description, "Scenario": scenario, "Note": "Rejected", "Polished Note": "Rejected", "Role": "Rejected"})
            
            # Respecting the number of needed approved notes
            if len(approved_notes) >= notes_count:
                break

    except Exception as e:
        print(f"An error occurred during note generation: {e}")
    
    finally:
        # Combine the results
        results = approved_notes + rejected_scenarios
        df = pd.DataFrame(results)
    return pd.DataFrame(results)

In [11]:
import datetime
current_date = datetime.datetime.now().strftime("%Y-%m-%d")

# Create the filename with the current date
path = "/Users/ahmadrezaie/papers/Synthetic_Data_Gen/data/output"



In [ ]:
""" 
disease_description = "Shortness of Breath"
notes_count = 5
df = generate_medical_notes(disease_description, notes_count)

# Save to CSV
df.to_csv(filename, index=False)
"""

In [12]:
sample_five_icds = ["HERPESVIRAL INFECTION, UNSPECIFIED", "SENSORINEURAL HEARING LOSS, BILATERAL", "CHRONIC VIRAL HEPATITIS C", 
                    "CHRONIC PAIN SYNDROME", "MAJOR DEPRESSIVE DISORDER, RECURRENT SEVERE WITHOUT PSYCHOTIC FEATURES"]

In [13]:
for condition in sample_five_icds:
    disease_description = condition
    filename = f"{path}/{disease_description}doctor_judge_notes_{current_date}_2.csv" 
    notes_count = 5 
    df = generate_medical_notes(disease_description, notes_count)
    df.head()

    # Save to CSV
    #df.to_csv(filename, index=False)

KeyboardInterrupt: 

In [72]:
# sampling notes for feedback from physicians

import os

feedback_dir = "/Users/ahmadrezaie/papers/Synthetic_Data_Gen/data/output/for_feedback_round1/after_polish"

# Initialize an empty DataFrame to hold samples
sampled_rows_df = pd.DataFrame()

for filename in os.listdir(feedback_dir):
    if filename.endswith(".csv"):
        file_path = os.path.join(feedback_dir, filename)
        df = pd.read_csv(file_path)
        # Sample one row from the top 5 rows randomly
        sampled_row = df.head(5).sample(1)
        # Append the sampled row to the DataFrame
        sampled_rows_df = pd.concat([sampled_rows_df, sampled_row], ignore_index=True)


sampled_rows_df.to_csv("/Users/ahmadrezaie/papers/Synthetic_Data_Gen/data/output/for_feedback_round1/after_polish/sample_notes.csv", sep="|")
sampled_rows_df

,Disease Description,Scenario,Note,Polished Note,Role
0,"SENSORINEURAL HEARING LOSS, BILATERAL",ROLE: Otologist (Ear Specialist)\n\n- Medical ...,**Medical Note**\n\n**Subjective:**\nThe patie...,**Medical Note**\n\n**Subjective:**\nThe patie...,Otologist (Ear Specialist)
1,CHRONIC PAIN SYNDROME,ROLE: Pain Management Specialist\n\n1) Medical...,Subjective:\nThe 42-year-old female patient pr...,"Subjective:\nThe patient, Emily, a 42-year-old...",Pain Management Specialist
2,CHRONIC VIRAL HEPATITIS C,ROLE: Infectious Disease Specialist\n\n- Medic...,Medical Note:\n\n1. Subjective:\nA 52-year-old...,Medical Note:\n\n1. Subjective:\nA 52-year-old...,Infectious Disease Specialist
3,"MAJOR DEPRESSIVE DISORDER, RECURRENT SEVERE WI...",ROLE: Psychiatrist\n\n1) Medical Outcome: Diag...,"Subjective:\nThe patient, a 38-year-old Caucas...","Subjective:\nThe patient, whom I shall refer t...",Psychiatrist
4,"HERPESVIRAL INFECTION, UNSPECIFIED",ROLE: Infectious Disease Specialist\n\n1) Medi...,Subjective:\nA 29-year-old female presents wit...,"Subjective:\nA 29-year-old female, whom we'll ...",Infectious Disease Specialist


In [1]:
import pandas as pd
df = pd.read_csv("/h/ahmad/SynthDataGen/Synthetic_Data_Gen/data/output/notes_onVector/CHRONIC PAIN SYNDROME_2024-05-15", sep="|")
df

,Disease Description,Scenario,Note,Polished Note,Role
0,CHRONIC PAIN SYNDROME,ROLE: Pain Management Specialist\n\n1) Medical...,**Subjective:** \nThe 45-year-old female patie...,"**Subjective:** \nThe patient, Ms. Jane Doe, a...",Pain Management Specialist
1,CHRONIC PAIN SYNDROME,ROLE: Pain Management Specialist\n\n** Medical...,**Subjective:**\nThe patient is a 45-year-old ...,"**Subjective:**\nThe patient, Jane Doe, a 45-y...",Pain Management Specialist
2,CHRONIC PAIN SYNDROME,ROLE: Pain Management Specialist\n\n1) Medical...,**Subjective:**\nThe patient is a 46-year-old ...,"**Subjective:**\nThe patient, Mrs. Jane Doe, i...",Pain Management Specialist
3,CHRONIC PAIN SYNDROME,ROLE: Pain Management Specialist\n\n1) Medical...,**Subjective:**\nThe patient is a 52-year-old ...,"**Subjective:**\nThe patient, Mrs. Linda Thomp...",Pain Management Specialist
4,CHRONIC PAIN SYNDROME,ROLE: Pain Management Specialist\n\n1) Medical...,**Subjective:**\n\nThe patient is a 45-year-ol...,"**Subjective:**\n\nThe patient, Mrs. Emily Tho...",Pain Management Specialist
5,CHRONIC PAIN SYNDROME,ROLE: Family Medicine Physician\n\n1) Medical ...,**Medical Note:**\n\n**Date:** [Insert Date of...,**Medical Note:**\n\n**Date:** [Insert Date of...,Family Medicine Physician
6,CHRONIC PAIN SYNDROME,ROLE: Pain Management Specialist\n\n1) Medical...,**Subjective:**\nThe patient is a 52-year-old ...,"**Subjective:**\nThe patient, Jane Doe, is a 5...",Pain Management Specialist
7,CHRONIC PAIN SYNDROME,ROLE: Family Medicine Physician\n\n1) Medical ...,**1. Subjective:**\nThe patient is a 45-year-o...,"**1. Subjective:**\nThe patient, Jane Doe, a 4...",Family Medicine Physician
8,CHRONIC PAIN SYNDROME,ROLE: Pain Management Specialist\n\n1) Medical...,**Subjective:**\nThe patient is a 45-year-old ...,"**Subjective:**\nThe patient, Jane Doe, is a 4...",Pain Management Specialist
9,CHRONIC PAIN SYNDROME,ROLE: Pain Management Specialist\n\n1) Medical...,**Subjective:**\nThe patient is a 48-year-old ...,"**Subjective:**\nThe patient, Mrs. Linda Thomp...",Pain Management Specialist


In [8]:
def get_embedding(text, model="text-embedding-3-small"):
   text = text.replace("\n", " ")
   return client.embeddings.create(input = [text], model=model).data[0].embedding

df = pd.read_csv("/h/ahmad/SynthDataGen/Synthetic_Data_Gen/data/output/notes_onVector/CHRONIC PAIN SYNDROME_2024-05-15", sep="|")
df['embedding'] = df.Scenario.apply(lambda x: get_embedding(x, model='text-embedding-3-small'))
df


,Disease Description,Scenario,Note,Polished Note,Role,embedding
0,CHRONIC PAIN SYNDROME,ROLE: Pain Management Specialist\n\n1) Medical...,**Subjective:** \nThe 45-year-old female patie...,"**Subjective:** \nThe patient, Ms. Jane Doe, a...",Pain Management Specialist,"[-0.005625685676932335, 0.026912622153759003, ..."
1,CHRONIC PAIN SYNDROME,ROLE: Pain Management Specialist\n\n** Medical...,**Subjective:**\nThe patient is a 45-year-old ...,"**Subjective:**\nThe patient, Jane Doe, a 45-y...",Pain Management Specialist,"[-0.015071071684360504, 0.02452823892235756, 0..."
2,CHRONIC PAIN SYNDROME,ROLE: Pain Management Specialist\n\n1) Medical...,**Subjective:**\nThe patient is a 46-year-old ...,"**Subjective:**\nThe patient, Mrs. Jane Doe, i...",Pain Management Specialist,"[0.006586750969290733, 0.025920720770955086, 0..."
3,CHRONIC PAIN SYNDROME,ROLE: Pain Management Specialist\n\n1) Medical...,**Subjective:**\nThe patient is a 52-year-old ...,"**Subjective:**\nThe patient, Mrs. Linda Thomp...",Pain Management Specialist,"[-0.007595145609229803, 0.024071602150797844, ..."
4,CHRONIC PAIN SYNDROME,ROLE: Pain Management Specialist\n\n1) Medical...,**Subjective:**\n\nThe patient is a 45-year-ol...,"**Subjective:**\n\nThe patient, Mrs. Emily Tho...",Pain Management Specialist,"[-0.00022266437008511275, 0.01821352168917656,..."
5,CHRONIC PAIN SYNDROME,ROLE: Family Medicine Physician\n\n1) Medical ...,**Medical Note:**\n\n**Date:** [Insert Date of...,**Medical Note:**\n\n**Date:** [Insert Date of...,Family Medicine Physician,"[-0.0019984173122793436, 0.018674982711672783,..."
6,CHRONIC PAIN SYNDROME,ROLE: Pain Management Specialist\n\n1) Medical...,**Subjective:**\nThe patient is a 52-year-old ...,"**Subjective:**\nThe patient, Jane Doe, is a 5...",Pain Management Specialist,"[-0.0014946127776056528, 0.02301930822432041, ..."
7,CHRONIC PAIN SYNDROME,ROLE: Family Medicine Physician\n\n1) Medical ...,**1. Subjective:**\nThe patient is a 45-year-o...,"**1. Subjective:**\nThe patient, Jane Doe, a 4...",Family Medicine Physician,"[-0.0065644304268062115, 0.012482728809118271,..."
8,CHRONIC PAIN SYNDROME,ROLE: Pain Management Specialist\n\n1) Medical...,**Subjective:**\nThe patient is a 45-year-old ...,"**Subjective:**\nThe patient, Jane Doe, is a 4...",Pain Management Specialist,"[0.0013666421873494983, 0.02832229994237423, 0..."
9,CHRONIC PAIN SYNDROME,ROLE: Pain Management Specialist\n\n1) Medical...,**Subjective:**\nThe patient is a 48-year-old ...,"**Subjective:**\nThe patient, Mrs. Linda Thomp...",Pain Management Specialist,"[-0.003352179192006588, 0.019855735823512077, ..."


In [14]:
from gen_utils import semantic_similarity_functions
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
mean_rejected_included= (df.embedding # extract 'embeddings' for each group
.apply(np.stack).reshape(-1, 1) # turns sequence of arrays into proper matrix
.apply(cosine_similarity) # the magic: compute pairwise similarity matrix
.apply(np.mean))

mean_rejected_included

AttributeError: 'Series' object has no attribute 'reshape'

In [32]:
df= pd.read_csv("/h/ahmad/SynthDataGen/Synthetic_Data_Gen/data/output/notes_onVector/CHRONIC PAIN SYNDROME_2024-05-15_with_embeding.csv", sep="|")
df.head()

,Disease Description,Scenario,Note,Polished Note,Role,embedding
0,CHRONIC PAIN SYNDROME,ROLE: Pain Management Specialist\n\n1) Medical...,**Subjective:** \nThe 45-year-old female patie...,"**Subjective:** \nThe patient, Ms. Jane Doe, a...",Pain Management Specialist,"[-0.0056213503703475, 0.027141816914081573, 0...."
1,CHRONIC PAIN SYNDROME,ROLE: Pain Management Specialist\n\n** Medical...,**Subjective:**\nThe patient is a 45-year-old ...,"**Subjective:**\nThe patient, Jane Doe, a 45-y...",Pain Management Specialist,"[-0.015071071684360504, 0.02452823892235756, 0..."
2,CHRONIC PAIN SYNDROME,ROLE: Pain Management Specialist\n\n1) Medical...,**Subjective:**\nThe patient is a 46-year-old ...,"**Subjective:**\nThe patient, Mrs. Jane Doe, i...",Pain Management Specialist,"[0.006586750969290733, 0.025920720770955086, 0..."
3,CHRONIC PAIN SYNDROME,ROLE: Pain Management Specialist\n\n1) Medical...,**Subjective:**\nThe patient is a 52-year-old ...,"**Subjective:**\nThe patient, Mrs. Linda Thomp...",Pain Management Specialist,"[-0.007615276146680117, 0.024071041494607925, ..."
4,CHRONIC PAIN SYNDROME,ROLE: Pain Management Specialist\n\n1) Medical...,**Subjective:**\n\nThe patient is a 45-year-ol...,"**Subjective:**\n\nThe patient, Mrs. Emily Tho...",Pain Management Specialist,"[-0.00022266437008511275, 0.01821352168917656,..."


In [31]:
import pandas as pd
import json
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity



# Convert JSON strings back to lists
df['embedding'] = df['embedding'].apply(json.loads)

# Ensure the column is of type float
df['embedding'] = df['embedding'].apply(lambda x: np.array(x, dtype=float))

# Stack embeddings into a 2D array
embeddings = np.vstack(df['embedding'].values)

# Compute cosine similarity
similarity_matrix = cosine_similarity(embeddings)

# Exclude self-similarities (diagonal elements)
np.fill_diagonal(similarity_matrix, np.nan)

# Calculate the average score, excluding NaN values
average_score = np.nanmean(similarity_matrix)

average_score

0.9229236558360676

In [33]:
pd.set_option('display.max_colwidth', None)

# Loop through the 'Polished Note' column and print its content
for note in df['Polished Note']:
    print(note)
    print("\n" + "-"*80 + "\n")  # Separator for better readability between notes

**Subjective:** 
The patient, Ms. Jane Doe, a 45-year-old female, presents with a history of chronic lower back pain, described as persistent with a severity of 7/10, lasting more than six months. The pain occurs daily and worsens with physical activity, significantly impacting her work and household tasks. Previous management with NSAIDs provided insufficient relief. She has a familial history of fibromyalgia and a personal history of a lower back sprain. Her hypertension is controlled with medication. She reports disrupted sleep due to pain, averaging 4-5 hours per night, a diet with insufficient fresh fruits and vegetables, and a sedentary lifestyle. She is a non-smoker with moderate alcohol use on weekends and denies recreational drug use. Ms. Doe is motivated to follow medical advice to improve her condition.

**Objective:** 
On examination, there is tenderness at the lumbar paraspinal muscles and reduced lumbar range of motion; however, there are no neurological deficits. The pat